# Density Sweep trên Subsampled ML-20M — M1 / M7 / R1  *(bản gọn, chạy thẳng)*

Notebook **tái dùng nguyên code & data gốc** của paper *LLM-MovieLens*, chỉ thêm lớp điều phối:
subsample theo mật độ → chạy **M1 / M7 / R1** → tổng hợp → vẽ **∆R1(d)**.

**Khớp code gốc để kiểm chứng:**
- **M1 / M7** ← `code/benchmark/run_experiment.py` (LightGCN / LightGCN-SF).
- **R1 (RLMRec-gene)** ← `external/RLMRec/encoder/train_encoder.py`.
- **Đánh giá R1 dùng cùng `evaluate.compute_metrics`** (full-ranking, mask train items, sum `layer_num`)
  như M1/M7 ⇒ NDCG@10/Recall@10/MRR so sánh trực tiếp.

**Sạch & an toàn:**
- Truyền đường dẫn qua **biến môi trường** (chạy được cả bản `run_experiment.py` cũ lẫn mới — *fix lỗi
  `unrecognized arguments: --data-dir`*).
- **Mọi output nằm trong `<repo>/experiments/density_sweep_sub20m/`** — không rải rác.
- R1 chạy trong *workdir riêng* (symlink) và **không sửa file lõi nào** (không patch `data_handler`,
  không ghi vào `data/`/`checkpoint/` của repo).

Chạy tuần tự từ trên xuống.

## 0 · Cấu hình

In [ ]:
import os, sys, json, time, shutil, subprocess, platform
from pathlib import Path

# ── Đường dẫn gốc ──────────────────────────────────────────────────────────────
def _autodetect_repo() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "code" / "benchmark" / "run_experiment.py").exists():
            return cand
    return here

REPO = _autodetect_repo()                 # ← sửa tay nếu cần: REPO = Path("/home/nd/Desktop")
BENCH = REPO / "code" / "benchmark"
RLMREC_ROOT = BENCH / "external" / "RLMRec"
FULL_PROCESSED_DIR = BENCH / "data" / "processed"

# ── Tự dò thư mục embedding (tránh lệch tên bge-large-en-v1.5 / bge-large-v1.5) ──
EMBEDDING_DIR_OVERRIDE = None             # đặt Path("...") nếu muốn ép cố định
def _autodetect_embedding_dir() -> Path:
    if EMBEDDING_DIR_OVERRIDE:
        return Path(EMBEDDING_DIR_OVERRIDE)
    out = BENCH.parent / "embedding_generator" / "output"
    if out.is_dir():
        cands = sorted(d for d in out.iterdir()
                       if d.is_dir() and (d / "profile_embeddings.npy").exists())
        if cands:
            return cands[0]
    return out / "bge-large-en-v1.5"      # fallback
EMBEDDING_DIR = _autodetect_embedding_dir()
ENCODER_NAME  = EMBEDDING_DIR.name        # dùng trong layout output (giống experiment_path repo)

# ── Bootstrap source (nếu thiếu) ────────────────────────────────────────────────
REPO_GIT_URL    = "https://github.com/anonyauthor4review-png/llm-movielens.git"
REPO_BRANCH     = None
LOCAL_REPO_ZIP  = None                    # vd "/content/llm-movielens-main.zip" (None = tự dò)
AUTO_PIP_INSTALL = True
FORCE_BOOTSTRAP = False                   # True = nạp lại source

# ── Thí nghiệm ──────────────────────────────────────────────────────────────────
DENSITIES = [600, 300, 150, 75]           # target int/item (10-core sẽ làm cao hơn → báo cáo achieved)
SEEDS     = [42, 123, 456]
METHODS   = ["M1", "M7", "R1"]
SUBSAMPLE_SEED = 42
RERUN = False                             # True = bỏ run_records, chạy lại tất cả
DEVICE = "auto"                           # "auto" | "cuda" | "mps" | "cpu"

# ── R1 (RLMRec-gene) hparams — protocol cùng-domain (mặc định upstream) ─────────
R1_DATASET        = "ml20m"               # TÁI DÙNG tên có sẵn ⇒ không patch data_handler
R1_LAYER_NUM      = 3
R1_MASK_RATIO     = 0.1
R1_RECON_WEIGHT   = 0.1
R1_RE_TEMPERATURE = 0.2
R1_REG_WEIGHT     = 1.0e-7
R1_EMBEDDING_SIZE = 32                     # đặt 128 nếu muốn so khớp dung lượng với M7

# ── Output: GÓI GỌN trong repo/experiments/density_sweep_sub20m/ ────────────────
EXP_ROOT     = REPO / "experiments" / "density_sweep_sub20m"
DATA_OUT_ROOT = EXP_ROOT / "data"          # subsample (processed_ml20m_sub{d})
CKPT_ROOT     = EXP_ROOT / "checkpoints"   # M1/M7/R1 best_model.pt + training_state.pt
RESULTS_ROOT  = EXP_ROOT / "results"       # results.json (M) + r1/seed-*-metrics.json
LOG_ROOT      = EXP_ROOT / "logs"
R1_DATA_ROOT  = EXP_ROOT / "r1_data"       # RLMRec-format pkl theo từng mật độ
R1_WORK_ROOT  = EXP_ROOT / "r1_work"       # workdir riêng (symlink encoder/data)
R1_MODELCONF  = EXP_ROOT / "r1_modelconf"
RUN_RECORDS   = EXP_ROOT / "run_records.jsonl"
for d in (DATA_OUT_ROOT, CKPT_ROOT, RESULTS_ROOT, LOG_ROOT, R1_DATA_ROOT, R1_WORK_ROOT, R1_MODELCONF):
    d.mkdir(parents=True, exist_ok=True)

def resolve_device(dev):
    if dev != "auto": return dev
    try:
        import torch
        if torch.cuda.is_available(): return "cuda"
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available(): return "mps"
    except Exception: pass
    return "cpu"
DEVICE = resolve_device(DEVICE)

print(f"REPO          = {REPO}")
print(f"EMBEDDING_DIR = {EMBEDDING_DIR}  (encoder={ENCODER_NAME})")
print(f"EXP_ROOT      = {EXP_ROOT}")
print(f"DEVICE        = {DEVICE}")
print(f"DENSITIES/SEEDS/METHODS = {DENSITIES} / {SEEDS} / {METHODS}")
print(f"R1: dataset='{R1_DATASET}'  embedding_size={R1_EMBEDDING_SIZE}")


## 0.5 · Bootstrap — tự động nạp source code repo (nếu thiếu)

Nếu thiếu source (`run_experiment.py`, `RLMRec`…), ô này tự nạp: (1) zip repo đã upload, hoặc
(2) `git clone`. Chỉ **thêm code**, **không đụng** `data/processed/` và `embedding_generator/output/`.

In [ ]:
def _need_source() -> bool:
    must = [BENCH / "run_experiment.py",
            RLMREC_ROOT / "encoder" / "train_encoder.py",
            RLMREC_ROOT / "encoder" / "data_utils" / "data_handler_general_cf.py",
            RLMREC_ROOT / "encoder" / "config" / "modelconf" / "lightgcn_gene.yml"]
    return not all(p.exists() for p in must)

def _find_clone_root(base: Path) -> Path:
    cands = [base, *[p for p in base.iterdir() if p.is_dir()]] if base.is_dir() else [base]
    for c in cands:
        if (c / "src" / "benchmark" / "run_experiment.py").exists(): return c
        if (c / "code" / "benchmark" / "run_experiment.py").exists(): return c
    raise FileNotFoundError(f"Không thấy src/benchmark hoặc code/benchmark trong {base}")

def _merge_source(clone_root: Path):
    src_bench = clone_root / "src" / "benchmark"
    if not src_bench.exists():
        src_bench = (clone_root / "code" / "benchmark").resolve()
    src_root = src_bench.parent
    BENCH.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src_bench, BENCH, dirs_exist_ok=True)
    eg = src_root / "embedding_generator"
    if eg.exists():
        shutil.copytree(eg, BENCH.parent / "embedding_generator", dirs_exist_ok=True)
    print(f"  ✓ merge source từ {src_bench}")

def _pip_if_missing():
    import importlib.util as u
    alias = {"yaml": "pyyaml", "sklearn": "scikit-learn"}
    need = [alias.get(m, m) for m in ["yaml","tqdm","matplotlib","scipy","sklearn","pandas","numpy"]
            if u.find_spec(m) is None]
    if need:
        print(f"  pip install {' '.join(need)} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=False)
    try:
        import torch  # noqa
    except Exception:
        print("  ⚠ torch chưa có — cài: pip install torch")

if FORCE_BOOTSTRAP or _need_source():
    print("Thiếu source — đang bootstrap...")
    got = False
    zips = ([Path(LOCAL_REPO_ZIP)] if LOCAL_REPO_ZIP else [])
    zips += sorted(Path("/content").glob("llm-movielens*.zip")) if Path("/content").exists() else []
    zips += sorted(Path.cwd().glob("llm-movielens*.zip"))
    for zp in zips:
        if zp.exists():
            print(f"  giải nén {zp} ...")
            tmp = Path("/tmp/_llmml_zip"); shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
            shutil.unpack_archive(str(zp), str(tmp))
            try: _merge_source(_find_clone_root(tmp)); got = True; break
            except FileNotFoundError as e: print(f"    (bỏ qua zip: {e})")
    if not got:
        tmp = Path("/tmp/_llmml_clone"); shutil.rmtree(tmp, ignore_errors=True)
        cmd = ["git", "clone", "--depth", "1"] + (["--branch", REPO_BRANCH] if REPO_BRANCH else []) + [REPO_GIT_URL, str(tmp)]
        print(f"  {' '.join(cmd)}")
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stderr[-800:])
            raise RuntimeError("git clone thất bại. Upload zip repo lên /content rồi chạy lại, "
                               "hoặc set LOCAL_REPO_ZIP/REPO_GIT_URL ở ô Cấu hình.")
        _merge_source(_find_clone_root(tmp))
    if AUTO_PIP_INSTALL: _pip_if_missing()
    print("Bootstrap xong." if not _need_source() else "⚠ Vẫn thiếu file — kiểm tra nguồn.")
else:
    print("Source đã có — bỏ qua bootstrap.")
    if AUTO_PIP_INSTALL: _pip_if_missing()


## 1 · Logging

In [ ]:
import logging
from datetime import datetime
MASTER_LOG = LOG_ROOT / f"master_{datetime.now():%Y%m%d-%H%M%S}.log"
logger = logging.getLogger("density_sweep"); logger.setLevel(logging.INFO); logger.handlers.clear()
_fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", "%H:%M:%S")
for h in (logging.StreamHandler(sys.stdout), logging.FileHandler(MASTER_LOG)):
    h.setFormatter(_fmt); logger.addHandler(h)
logger.propagate = False
logger.info(f"Master log → {MASTER_LOG}")


## 2 · Kiểm tra môi trường & khả năng `run_experiment.py`

Xác nhận data + embedding + scripts; và **tự dò cách truyền đường dẫn**: bản `run_experiment.py` mới
hỗ trợ cờ `--data-dir/...`, bản cũ thì không — nhưng cả hai đều đọc **biến môi trường**
`DATA_DIR/EMBEDDING_DIR/RESULTS_DIR/CHECKPOINT_DIR` (qua `config.py`). Notebook luôn truyền bằng ENV
và chỉ thêm cờ CLI nếu bản hỗ trợ.

In [ ]:
def _ok(b): return "✓" if b else "✗"
problems = []

# scripts
RUN_EXP   = BENCH / "run_experiment.py"
TRAIN_ENC = RLMREC_ROOT / "encoder" / "train_encoder.py"
CONFIG_PY = BENCH / "config.py"
LIGHTGCN_GENE_YML = RLMREC_ROOT / "encoder" / "config" / "modelconf" / "lightgcn_gene.yml"
for p in (RUN_EXP, TRAIN_ENC, CONFIG_PY, LIGHTGCN_GENE_YML):
    print(f"  {_ok(p.exists())} {p}")
    if not p.exists(): problems.append(str(p))

# data
print("\n[data ML-20M processed]")
for f in ["train.csv","val.csv","test.csv","item_map.json","stats.json"]:
    p = FULL_PROCESSED_DIR / f
    print(f"  {_ok(p.exists())} {p}")
    if not p.exists(): problems.append(str(p))

# embedding (đã tự dò)
print(f"\n[embedding: {EMBEDDING_DIR}]")
import numpy as _np
for f in ["profile_embeddings.npy","mood_vectors.npy","movie_id_index.json"]:
    p = EMBEDDING_DIR / f
    if not p.exists():   # thử symlink từ thư mục cha
        cand = EMBEDDING_DIR.parent / f
        if cand.exists():
            try: p.symlink_to(cand); print(f"  ↪ symlink {f}")
            except Exception: pass
    ok = p.exists()
    shape = (_np.load(p).shape if (ok and f.endswith('.npy')) else
             (f"({len(json.loads(p.read_text()))} ids)" if ok else "—"))
    print(f"  {_ok(ok)} {p}  {shape}")
    if not ok: problems.append(str(p))

# khả năng truyền path của run_experiment.py
print("\n[khả năng run_experiment.py]")
SUPPORTS_CLI_DIRS = False; SUPPORTS_ENV_DIRS = False
try:
    h = subprocess.run([sys.executable, "run_experiment.py", "--help"],
                       cwd=str(BENCH), capture_output=True, text=True, timeout=120)
    helptext = (h.stdout + h.stderr)
    SUPPORTS_CLI_DIRS = "--data-dir" in helptext
except Exception as e:
    print(f"  (không gọi được --help: {e})")
try:
    cfgtxt = CONFIG_PY.read_text()
    SUPPORTS_ENV_DIRS = all(f'environ.get("{k}"' in cfgtxt or f"environ.get('{k}'" in cfgtxt
                            for k in ["DATA_DIR","RESULTS_DIR","CHECKPOINT_DIR","EMBEDDING_DIR"])
except Exception:
    pass
print(f"  cờ CLI --data-dir: {_ok(SUPPORTS_CLI_DIRS)}   |   biến môi trường (config.py): {_ok(SUPPORTS_ENV_DIRS)}")
if not (SUPPORTS_CLI_DIRS or SUPPORTS_ENV_DIRS):
    problems.append("run_experiment.py không nhận path qua CLI lẫn ENV — hãy đặt FORCE_BOOTSTRAP=True "
                    "để refresh source rồi chạy lại.")

print()
if problems:
    logger.warning("THIẾU/ VẤN ĐỀ %d mục:", len(problems))
    for x in problems: logger.warning("   - %s", x)
else:
    logger.info("Môi trường OK. Truyền path qua ENV%s." %
                (" + cờ CLI" if SUPPORTS_CLI_DIRS else ""))


## 3 · Subsample

Sinh `experiments/.../data/processed_ml20m_sub{d}/`. **`item_map.json` dùng khoá `ml20m_movieId`**
để `FeatureLoader` căn đúng embedding (note #1 trong README). Cột `emb_matched` phải ≈ `n_items`.

In [ ]:
import numpy as np
import pandas as pd

def _kcore(df, k=10):
    while True:
        uc = df.groupby("userId").size(); ic = df.groupby("movieId").size()
        new = df[df.userId.isin(uc[uc>=k].index) & df.movieId.isin(ic[ic>=k].index)]
        if len(new) == len(df): return new
        df = new

def subsample_density(target, out_dir, seed=SUBSAMPLE_SEED):
    out_dir.mkdir(parents=True, exist_ok=True)
    train = pd.read_csv(FULL_PROCESSED_DIR/"train.csv")
    val   = pd.read_csv(FULL_PROCESSED_DIR/"val.csv")
    test  = pd.read_csv(FULL_PROCESSED_DIR/"test.csv")
    base_map = json.loads((FULL_PROCESSED_DIR/"item_map.json").read_text())
    proc_to_ml20m = {int(v): int(k) for k, v in base_map.items()}

    target_total = target * train.movieId.nunique()
    rng = np.random.default_rng(seed)
    u_int = train.groupby("userId").size()
    users = u_int.index.values.copy(); rng.shuffle(users)
    cum = u_int.loc[users].cumsum().values
    n_keep = min(int(np.searchsorted(cum, target_total) + 1), len(users))
    kept = set(users[:n_keep].tolist())

    train = train[train.userId.isin(kept)].reset_index(drop=True)
    val   = val[val.userId.isin(kept)].reset_index(drop=True)
    test  = test[test.userId.isin(kept)].reset_index(drop=True)
    train = _kcore(train, 10).reset_index(drop=True)
    items_kept = set(train.movieId.unique().tolist()); users_kept = set(train.userId.unique().tolist())
    val  = val[val.userId.isin(users_kept) & val.movieId.isin(items_kept)].reset_index(drop=True)
    test = test[test.userId.isin(users_kept) & test.movieId.isin(items_kept)].reset_index(drop=True)

    item_old_to_new = {int(o): i for i, o in enumerate(sorted(items_kept))}
    user_old_to_new = {int(o): i for i, o in enumerate(sorted(users_kept))}
    for df in (train, val, test):
        df["userId"]  = df["userId"].map(user_old_to_new).astype("int64")
        df["movieId"] = df["movieId"].map(item_old_to_new).astype("int64")

    sub_to_ml20m = {str(new): proc_to_ml20m[old] for old, new in item_old_to_new.items() if old in proc_to_ml20m}
    item_map_loader = {str(proc_to_ml20m[old]): new for old, new in item_old_to_new.items() if old in proc_to_ml20m}

    train.to_csv(out_dir/"train.csv", index=False)
    val.to_csv(out_dir/"val.csv", index=False)
    test.to_csv(out_dir/"test.csv", index=False)
    (out_dir/"item_map.json").write_text(json.dumps(item_map_loader))
    (out_dir/"user_map.json").write_text(json.dumps({str(k): v for k, v in user_old_to_new.items()}))
    (out_dir/"subsampled_to_ml20m_movieId.json").write_text(json.dumps(sub_to_ml20m))

    n_u = int(train.userId.max()+1); n_i = int(train.movieId.max()+1)
    counts = train.groupby("movieId").size(); achieved = float(len(train)/n_i)
    stats = {"n_users":n_u,"n_items":n_i,"n_train":int(len(train)),"n_val":int(len(val)),"n_test":int(len(test)),
             "density":float(len(train)/(n_u*n_i)),"n_cold_items":int((counts<10).sum()),
             "n_medium_items":int(((counts>=10)&(counts<50)).sum()),"n_warm_items":int((counts>=50).sum()),
             "target_int_per_item":target,"achieved_int_per_item":achieved,
             "items_per_user":float(len(train)/n_u),"emb_matched":len(sub_to_ml20m),"subsample_seed":seed}
    (out_dir/"stats.json").write_text(json.dumps(stats, indent=2))
    return stats

DENSITY_DIRS = {}; rows = []
for d in DENSITIES:
    out_dir = DATA_OUT_ROOT / f"processed_ml20m_sub{d}"; DENSITY_DIRS[d] = out_dir
    need = ["train.csv","val.csv","test.csv","item_map.json","stats.json","subsampled_to_ml20m_movieId.json"]
    if (not RERUN) and all((out_dir/f).exists() for f in need):
        stats = json.loads((out_dir/"stats.json").read_text())
        logger.info(f"[sub{d}] đã có (achieved={stats['achieved_int_per_item']:.1f})")
    else:
        logger.info(f"[sub{d}] subsample target={d} ...")
        stats = subsample_density(d, out_dir)
        logger.info(f"[sub{d}] users={stats['n_users']:,} items={stats['n_items']:,} "
                    f"train={stats['n_train']:,} achieved={stats['achieved_int_per_item']:.1f} "
                    f"emb_matched={stats['emb_matched']}/{stats['n_items']}")
    rows.append({"target":d,"n_users":stats["n_users"],"n_items":stats["n_items"],"n_train":stats["n_train"],
                 "achieved_int_per_item":round(stats["achieved_int_per_item"],1),
                 "items_per_user":round(stats["items_per_user"],1),"emb_matched":stats["emb_matched"]})
print(); print(pd.DataFrame(rows).set_index("target").to_string())


## 4 · Runner M (M1 / M7)

Gọi `run_experiment.py` và **truyền path qua biến môi trường** (`DATA_DIR/EMBEDDING_DIR/RESULTS_DIR/
CHECKPOINT_DIR`) — chạy được cả bản cũ lẫn mới. Nếu bản hỗ trợ cờ CLI thì thêm cờ cho tường minh.
Sau khi chạy, kiểm tra `results.json` đã nằm đúng `experiments/.../results/sub{d}/...`.

In [ ]:
M_SPEC = {"M1": ("lightgcn","none"), "M7": ("lightgcn_sf","llm_prof_mood")}
CFG_LOWER = {"M1":"m1","M7":"m7","R1":"r1"}

def run_M_cell(method, d, seed):
    model, features = M_SPEC[method]
    data_dir    = DENSITY_DIRS[d]
    results_dir = RESULTS_ROOT / f"sub{d}"
    ckpt_dir    = CKPT_ROOT / f"sub{d}"
    log_dir     = LOG_ROOT / f"sub{d}"; log_dir.mkdir(parents=True, exist_ok=True)
    log_path    = log_dir / f"{method}-seed{seed}.log"

    env = os.environ.copy()
    env.update({"DATA_DIR":str(data_dir), "EMBEDDING_DIR":str(EMBEDDING_DIR),
                "RESULTS_DIR":str(results_dir), "CHECKPOINT_DIR":str(ckpt_dir)})
    cmd = [sys.executable, "run_experiment.py",
           "--model", model, "--features", features, "--seed", str(seed), "--device", DEVICE]
    if SUPPORTS_CLI_DIRS:
        cmd += ["--data-dir",str(data_dir), "--embedding-dir",str(EMBEDDING_DIR),
                "--results-dir",str(results_dir), "--checkpoint-dir",str(ckpt_dir)]

    t0 = time.time()
    with open(log_path, "w") as lf:
        proc = subprocess.run(cmd, cwd=str(BENCH), env=env, stdout=lf, stderr=subprocess.STDOUT, text=True)
    wall = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"{method} sub{d} seed{seed} FAILED (exit={proc.returncode}); xem {log_path}")

    exp_name = f"{CFG_LOWER[method]}/{ENCODER_NAME}/seed-{seed}"
    res_json = results_dir / exp_name / "results.json"
    if not res_json.exists():
        raise RuntimeError(f"Không thấy {res_json}. Có thể bản run_experiment.py không nhận path qua ENV; "
                           f"đặt FORCE_BOOTSTRAP=True để refresh source rồi chạy lại. Log: {log_path}")
    res = json.loads(res_json.read_text()); tm = res.get("test_metrics", {})
    ckpt = ckpt_dir / exp_name / "best_model.pt"
    train_time = wall
    try:
        import torch
        st = torch.load(ckpt_dir/exp_name/"training_state.pt", map_location="cpu", weights_only=False)
        train_time = float(sum(e.get("train_time",0.0) for e in st.get("history",[])))
    except Exception: pass
    return {"ndcg10":tm.get("NDCG@10"),"recall10":tm.get("Recall@10"),"mrr":tm.get("MRR"),
            "ndcg20":tm.get("NDCG@20"),"recall20":tm.get("Recall@20"),
            "best_epoch":res.get("best_epoch"),"train_time_sec":round(train_time,1),
            "ckpt_path":str(ckpt)}


## 5 · Runner R1 (RLMRec-gene) — sạch, không sửa lõi

Mỗi ô R1: (1) dựng data RLMRec-format vào `experiments/.../r1_data/sub{d}/`; (2) dựng **workdir riêng**
(symlink `encoder` → repo, `data/ml20m` → `r1_data/sub{d}`) nên tái dùng được tên dataset có sẵn
**`ml20m`** ⇒ *không patch* `data_handler`, *không* ghi vào repo; (3) train bằng `train_encoder.py`
với modelconf override (`ckpt_out_dir`/`resume` để có `best_model.pt`+`training_state.pt`,
`save_model:false` để không rải `.pth` vào repo); (4) **eval bằng `compute_metrics`** từ `best_model.pt`.

In [ ]:
import pickle, scipy.sparse as sp
from scipy.sparse import csr_matrix
from collections import defaultdict

ENC_DIR = RLMREC_ROOT / "encoder"
_BENCH = str(BENCH)
if _BENCH not in sys.path: sys.path.insert(0, _BENCH)

# (1) prepare RLMRec-format data
def _build_sparse(csv, n_u, n_i):
    df = pd.read_csv(csv)
    return csr_matrix((np.ones(len(df),np.float32),(df.userId.values,df.movieId.values)), shape=(n_u,n_i))
def prepare_r1_data(d):
    data_dir = DENSITY_DIRS[d]; out = R1_DATA_ROOT / f"sub{d}"; out.mkdir(parents=True, exist_ok=True)
    st = json.loads((data_dir/"stats.json").read_text()); n_u,n_i = st["n_users"],st["n_items"]
    pickle.dump(_build_sparse(data_dir/"train.csv",n_u,n_i),(out/"trn_mat.pkl").open("wb"))
    pickle.dump(_build_sparse(data_dir/"val.csv",  n_u,n_i),(out/"val_mat.pkl").open("wb"))
    pickle.dump(_build_sparse(data_dir/"test.csv", n_u,n_i),(out/"tst_mat.pkl").open("wb"))
    prof = np.load(EMBEDDING_DIR/"profile_embeddings.npy")
    ml_ids = json.loads((EMBEDDING_DIR/"movie_id_index.json").read_text())
    sub2ml = json.loads((data_dir/"subsampled_to_ml20m_movieId.json").read_text())
    row_of = {int(m): r for r, m in enumerate(ml_ids)}
    item_emb = np.zeros((n_i, prof.shape[1]), np.float32); matched = 0
    for s, ml in sub2ml.items():
        r = row_of.get(int(ml))
        if r is not None: item_emb[int(s)] = prof[r]; matched += 1
    pickle.dump(item_emb,(out/"itm_emb_np.pkl").open("wb"))
    tdf = pd.read_csv(data_dir/"train.csv"); usr = np.zeros((n_u,item_emb.shape[1]),np.float32)
    for uid,g in tdf.groupby("userId"): usr[uid] = item_emb[g.movieId.values].mean(0)
    nrm = np.linalg.norm(usr,axis=1,keepdims=True); nrm[nrm==0]=1.0
    pickle.dump((usr/nrm).astype(np.float32),(out/"usr_emb_np.pkl").open("wb"))
    pickle.dump([{"profile":"","reasoning":""}]*n_i,(out/"itm_prf.pkl").open("wb"))
    pickle.dump([{"profile":"","reasoning":""}]*n_u,(out/"usr_prf.pkl").open("wb"))
    logger.info(f"[R1 prep sub{d}] aligned {matched}/{n_i} items → {out.relative_to(REPO)}")
    return out

# (2) workdir riêng với symlink encoder + data (fallback copy nếu symlink lỗi)
def _link_or_copy(target: Path, link: Path):
    if link.exists() or link.is_symlink():
        if link.is_symlink() or link.is_file(): link.unlink()
        else: shutil.rmtree(link)
    try: link.symlink_to(target, target_is_directory=target.is_dir())
    except Exception: shutil.copytree(target, link) if target.is_dir() else shutil.copy2(target, link)
def setup_r1_workdir(d):
    work = R1_WORK_ROOT / f"sub{d}"; (work/"data").mkdir(parents=True, exist_ok=True)
    _link_or_copy(ENC_DIR, work/"encoder")
    _link_or_copy(R1_DATA_ROOT / f"sub{d}", work/"data"/R1_DATASET)  # data/ml20m → r1_data/sub{d}
    return work

# (3) modelconf override (tái dùng block model.ml20m; ckpt_out_dir/resume; save_model:false)
def write_r1_modelconf(d, seed):
    import yaml
    base = yaml.safe_load(LIGHTGCN_GENE_YML.read_text())
    base["model"]["name"] = "lightgcn_gene"; base["model"]["embedding_size"] = R1_EMBEDDING_SIZE
    base["model"][R1_DATASET] = {"layer_num":R1_LAYER_NUM,"reg_weight":R1_REG_WEIGHT,
                                 "mask_ratio":R1_MASK_RATIO,"recon_weight":R1_RECON_WEIGHT,
                                 "re_temperature":R1_RE_TEMPERATURE}
    base["data"]["name"] = R1_DATASET
    ckpt_out = CKPT_ROOT / f"sub{d}" / f"r1/{ENCODER_NAME}/seed-{seed}"; ckpt_out.mkdir(parents=True, exist_ok=True)
    base["train"]["seed"] = seed
    base["train"]["save_model"] = False                 # không rải .pth vào repo
    base["train"]["ckpt_out_dir"] = str(ckpt_out)       # best_model.pt + training_state.pt
    base["train"]["resume"] = True
    yml = R1_MODELCONF / f"lightgcn_gene_sub{d}_seed{seed}.yml"
    yml.write_text(yaml.safe_dump(base, sort_keys=False, allow_unicode=True))
    return yml, ckpt_out

def train_R1(d, seed, log_path):
    work = setup_r1_workdir(d); yml, ckpt_out = write_r1_modelconf(d, seed)
    cmd = [sys.executable, "encoder/train_encoder.py", "--model","lightgcn_gene",
           "--dataset", R1_DATASET, "--model-conf", str(yml), "--device", DEVICE, "--seed", str(seed)]
    t0 = time.time()
    with open(log_path, "w") as lf:
        proc = subprocess.run(cmd, cwd=str(work), env=os.environ.copy(),
                              stdout=lf, stderr=subprocess.STDOUT, text=True)
    wall = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"R1 sub{d} seed{seed} train FAILED (exit={proc.returncode}); xem {log_path}")
    return wall, ckpt_out

# (4) eval bằng compute_metrics (đọc best_model.pt → user_embeds/item_embeds)
def _norm_adj(n_u, n_i, train_user_items, device):
    import torch
    rows, cols = [], []
    for u, its in train_user_items.items():
        for it in its: rows.append(u); cols.append(it)
    inter = csr_matrix((np.ones(len(rows),np.float32),(rows,cols)), shape=(n_u,n_i))
    bip = sp.vstack([sp.hstack([csr_matrix((n_u,n_u)), inter]),
                     sp.hstack([inter.T, csr_matrix((n_i,n_i))])]).tocsr()
    bip = (bip != 0).astype(np.float32)
    deg = np.array(bip.sum(1)).flatten()
    dis = np.power(deg, -0.5, where=deg>0, out=np.zeros_like(deg)); dis[np.isinf(dis)] = 0.0
    coo = (sp.diags(dis) @ bip @ sp.diags(dis)).tocoo()
    idx = torch.LongTensor(np.vstack([coo.row, coo.col])); val = torch.FloatTensor(coo.data)
    return torch.sparse_coo_tensor(idx, val, coo.shape, device=device).coalesce()

def eval_R1(d, seed, ckpt_out):
    import torch
    from data.dataset import InteractionData
    from evaluate import compute_metrics
    TOP_K = [10,20,50]
    data = InteractionData(data_dir=DENSITY_DIRS[d])
    adj = _norm_adj(data.n_users, data.n_items, data.train_user_items, DEVICE)
    sd = torch.load(ckpt_out/"best_model.pt", map_location=DEVICE, weights_only=False)
    ue = sd["user_embeds"].to(DEVICE); ie = sd["item_embeds"].to(DEVICE)
    emb = torch.cat([ue, ie], 0); lst = [emb]
    for _ in range(R1_LAYER_NUM):
        emb = torch.sparse.mm(adj, lst[-1]); lst.append(emb)
    fin = sum(lst); uf, itf = fin[:ue.shape[0]], fin[ue.shape[0]:]
    eval_users = sorted(data.test_user_items.keys()); all_m = defaultdict(list)
    with torch.no_grad():
        for s in range(0, len(eval_users), 256):
            bu = eval_users[s:s+256]; ur = uf[torch.LongTensor(bu).to(DEVICE)]
            scores = (ur @ itf.T).cpu().numpy()
            for i, uid in enumerate(bu):
                gt = data.test_user_items.get(uid, set())
                if not gt: continue
                tr = data.train_user_items.get(uid, set())
                for k, v in compute_metrics(scores[i], gt, tr, data.n_items, TOP_K).items():
                    all_m[k].append(v)
    m = {k: float(np.mean(v)) for k, v in all_m.items()}
    best_epoch = None
    try: best_epoch = int(torch.load(ckpt_out/"training_state.pt", map_location="cpu", weights_only=False).get("best_epoch"))
    except Exception: pass
    out = {"ndcg10":m.get("NDCG@10"),"recall10":m.get("Recall@10"),"mrr":m.get("MRR"),
           "ndcg20":m.get("NDCG@20"),"recall20":m.get("Recall@20"),"best_epoch":best_epoch}
    rd = RESULTS_ROOT / f"sub{d}" / "r1"; rd.mkdir(parents=True, exist_ok=True)
    (rd / f"seed-{seed}-metrics.json").write_text(json.dumps(out, indent=2))
    return out

def run_R1_cell(d, seed):
    log_dir = LOG_ROOT / f"sub{d}"; log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"R1-seed{seed}.log"
    prepare_r1_data(d)
    wall, ckpt_out = train_R1(d, seed, log_path)
    out = eval_R1(d, seed, ckpt_out)
    out["train_time_sec"] = round(wall, 1); out["ckpt_path"] = str(ckpt_out/"best_model.pt")
    return out


## 6 · Sweep (4 × 3 × 3 = 36 ô)

Idempotent: ô đã `ok` trong `run_records.jsonl` được bỏ qua (trừ khi `RERUN=True`). Resume được.

In [ ]:
def _load_records():
    recs = {}
    if RUN_RECORDS.exists():
        for line in RUN_RECORDS.read_text().splitlines():
            if line.strip():
                r = json.loads(line); recs[(r["density"],r["method"],r["seed"])] = r
    return recs
def _append(rec):
    with open(RUN_RECORDS, "a") as f: f.write(json.dumps(rec, default=str)+"\n")

records = {} if RERUN else _load_records()
if RERUN and RUN_RECORDS.exists(): RUN_RECORDS.unlink()
total = len(DENSITIES)*len(METHODS)*len(SEEDS); done = 0
logger.info(f"=== SWEEP: {total} ô (device={DEVICE}) ===")
for d in DENSITIES:
    for method in METHODS:
        for seed in SEEDS:
            key = (d, method, seed); done += 1
            if (not RERUN) and records.get(key, {}).get("status") == "ok":
                logger.info(f"[{done}/{total}] sub{d} {method} seed{seed} — đã ok, bỏ qua"); continue
            logger.info(f"[{done}/{total}] === sub{d} {method} seed{seed} ===")
            t0 = time.time()
            try:
                metrics = run_M_cell(method, d, seed) if method in M_SPEC else run_R1_cell(d, seed)
                ach = json.loads((DENSITY_DIRS[d]/"stats.json").read_text())["achieved_int_per_item"]
                rec = {"density":d,"achieved_int_per_item":round(ach,1),"method":method,"seed":seed,
                       "status":"ok","elapsed_sec":round(time.time()-t0,1), **metrics}
                logger.info(f"    NDCG@10={metrics['ndcg10']:.4f} Recall@10={metrics['recall10']:.4f} "
                            f"MRR={metrics['mrr']:.4f} ({rec['elapsed_sec']}s)")
            except Exception as e:
                rec = {"density":d,"method":method,"seed":seed,"status":"error","error":str(e),
                       "elapsed_sec":round(time.time()-t0,1)}
                logger.error(f"    LỖI: {e}")
            records[key] = rec; _append(rec)
logger.info("=== SWEEP xong ===")


## 7 · Tổng hợp → `per_run_results.csv`, `summary_mean_std.csv`, `delta_R1_vs_M7.csv`

In [ ]:
import pandas as pd, numpy as np
recs = [json.loads(l) for l in RUN_RECORDS.read_text().splitlines() if l.strip()]
latest = {}
for r in recs: latest[(r["density"],r["method"],r["seed"])] = r
df = pd.DataFrame([r for r in latest.values() if r.get("status")=="ok"])
if df.empty:
    print("Chưa có ô 'ok' — chạy ô Sweep trước.")
else:
    cols = ["density","achieved_int_per_item","method","seed","ndcg10","recall10","mrr",
            "ndcg20","recall20","best_epoch","train_time_sec","ckpt_path"]
    per_run = df[[c for c in cols if c in df.columns]].sort_values(["density","method","seed"])
    per_run.to_csv(EXP_ROOT/"per_run_results.csv", index=False)
    g = per_run.groupby(["density","achieved_int_per_item","method"])
    summary = g.agg(ndcg10_mean=("ndcg10","mean"), ndcg10_std=("ndcg10", lambda x:x.std(ddof=1)),
                    recall10_mean=("recall10","mean"), recall10_std=("recall10", lambda x:x.std(ddof=1)),
                    mrr_mean=("mrr","mean"), mrr_std=("mrr", lambda x:x.std(ddof=1)),
                    n_seeds=("seed","count")).reset_index().sort_values(["density","method"])
    summary.to_csv(EXP_ROOT/"summary_mean_std.csv", index=False)
    drows = []
    for (d,ach), sub in summary.groupby(["density","achieved_int_per_item"]):
        m7 = sub[sub.method=="M7"]["ndcg10_mean"]; r1 = sub[sub.method=="R1"]["ndcg10_mean"]
        if len(m7) and len(r1) and float(m7.iloc[0])!=0:
            m7v,r1v = float(m7.iloc[0]),float(r1.iloc[0])
            drows.append({"density":d,"achieved_int_per_item":ach,"M7_ndcg10":round(m7v,4),
                          "R1_ndcg10":round(r1v,4),"delta_R1_vs_M7_pct":round((r1v-m7v)/m7v*100,2)})
    delta = pd.DataFrame(drows)
    if not delta.empty:
        delta = delta.sort_values("achieved_int_per_item", ascending=False)
        delta.to_csv(EXP_ROOT/"delta_R1_vs_M7.csv", index=False)
    print("── per_run_results.csv ──"); print(per_run.to_string(index=False))
    print("\n── summary_mean_std.csv ──"); print(summary.to_string(index=False))
    print("\n── delta_R1_vs_M7.csv ──")
    print(delta.to_string(index=False) if not delta.empty
          else "(chưa đủ cặp M7+R1 cùng mật độ để tính ∆)")


## 8 · Vẽ ∆R1(d)

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, numpy as np, pandas as pd
sp_ = EXP_ROOT/"summary_mean_std.csv"; dp_ = EXP_ROOT/"delta_R1_vs_M7.csv"
if not sp_.exists():
    print("Chưa có summary — chạy ô Tổng hợp trước.")
else:
    summary = pd.read_csv(sp_)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13,5))
    if dp_.exists() and len(pd.read_csv(dp_)):
        delta = pd.read_csv(dp_).sort_values("achieved_int_per_item")
        x = np.log(delta["achieved_int_per_item"].values); y = delta["delta_R1_vs_M7_pct"].values
        ax1.axhline(0, color="grey", lw=0.8, ls="--"); ax1.plot(x, y, "o-", color="#c0392b", lw=2, ms=8)
        for xi, yi, di in zip(x, y, delta["achieved_int_per_item"].values):
            ax1.annotate(f"{yi:+.1f}%\n(d≈{di:.0f})", (xi, yi), textcoords="offset points",
                         xytext=(0,8), ha="center", fontsize=8)
    ax1.set_xlabel("log(achieved int/item)"); ax1.set_ylabel("∆R1 vs M7 (% NDCG@10)")
    ax1.set_title("∆R1(d): mật độ thấp → R1 thắng?"); ax1.grid(alpha=0.3)
    colors = {"M1":"#2980b9","M7":"#27ae60","R1":"#c0392b"}
    for method, sub in summary.groupby("method"):
        sub = sub.sort_values("achieved_int_per_item"); x = np.log(sub["achieved_int_per_item"].values)
        ax2.errorbar(x, sub["ndcg10_mean"].values, yerr=sub["ndcg10_std"].values, marker="o",
                     capsize=3, lw=2, label=method, color=colors.get(method))
    ax2.set_xlabel("log(achieved int/item)"); ax2.set_ylabel("NDCG@10 (mean ± std)")
    ax2.set_title("NDCG@10 theo mật độ"); ax2.legend(); ax2.grid(alpha=0.3)
    fig.tight_layout(); out = EXP_ROOT/"density_sweep_plot.png"
    fig.savefig(out, dpi=150, bbox_inches="tight"); print(f"Đã lưu {out}")
    try:
        from IPython.display import Image, display; display(Image(filename=str(out)))
    except Exception: pass
